# Model 1: CatBoost

In [2]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from src.preprocessing import PreprocessConfig, preprocess_train_test

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 250)

RANDOM_STATE = 42

In [4]:
# Load raw → preprocess for CatBoost

TRAIN_PATH = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Train.csv"
TEST_PATH  = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

cfg = PreprocessConfig(id_col="ID", target_col="Target")

train, test = preprocess_train_test(train_raw, test_raw, cfg, for_model="catboost")

TARGET = cfg.target_col
ID = cfg.id_col

print("train:", train.shape, "| test:", test.shape)
train.head()

train: (9618, 47) | test: (2405, 46)


,ID,country,owner_age,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,personal_income,business_expenses,business_turnover,business_age_years,motor_vehicle_insurance,has_mobile_money,current_problem_cash_flow,has_cellphone,owner_sex,offers_credit_to_customers,attitude_satisfied_with_achievement,has_credit_card,keeps_financial_records,perception_insurance_companies_dont_insure_businesses_like_yours,perception_insurance_important,has_insurance,covid_essential_service,attitude_more_successful_next_year,problem_sourcing_money,marketing_word_of_mouth,has_loan_account,has_internet_banking,has_debit_card,future_risk_theft_stock,business_age_months,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,Target,personal_income_missing,business_expenses_missing,business_turnover_missing,log_personal_income,log_business_expenses,log_business_turnover,business_age_total_months,business_age_months_ge_12
0,ID_3CFL0U,eswatini,63.0,yes,no,no,no,yes,3000.0,6000.0,7000.0,14.0,never had,have now,yes,yes,male,"yes, sometimes",no,never had,"yes, always",yes,yes,no,yes,yes,yes,yes,never had,never had,never had,NaN,6.0,never had,used to have but don't have now,NaN,never had,never had,Low,0,0,0,8.006701,8.699681,8.853808,174.0,0
1,ID_XWI7G3,zimbabwe,39.0,no,yes,yes,no,yes,NaN,NaN,NaN,15.0,have now,have now,NaN,yes,male,"yes, sometimes",yes,never had,"yes, always",no,yes,yes,yes,NaN,NaN,NaN,NaN,NaN,NaN,no,3.0,never had,never had,NaN,NaN,NaN,Medium,1,1,1,NaN,NaN,NaN,183.0,0
2,ID_TY93LV,malawi,34.0,don't know,no,no,don't know,yes,30000.0,6000.0,13000.0,5.0,NaN,never had,yes,yes,male,"yes, sometimes",yes,never had,no,don't know,yes,no,NaN,yes,yes,no,never had,never had,never had,yes,NaN,NaN,NaN,yes,NaN,NaN,Low,0,0,0,10.308986,8.699681,9.472782,60.0,0
3,ID_9OP2C8,malawi,28.0,yes,no,no,no,no,180000.0,60000.0,30000.0,1.0,NaN,have now,no,yes,female,"yes, sometimes",no,never had,no,no,yes,no,NaN,yes,no,no,never had,never had,never had,no,NaN,NaN,NaN,yes,never had,have now,Low,0,0,0,12.100718,11.002117,10.308986,12.0,0
4,ID_13REYS,zimbabwe,43.0,yes,no,no,yes,yes,50.0,2400.0,1800.0,3.0,never had,NaN,NaN,no,female,"yes, sometimes",yes,never had,no,yes,yes,no,no,NaN,NaN,NaN,NaN,NaN,NaN,no,0.0,never had,never had,yes,NaN,NaN,Low,0,0,0,3.931826,7.783641,7.496097,36.0,0


In [5]:
# Define X/y and categorical feature indices

y = train[TARGET]
X = train.drop(columns=[TARGET])

# Identify categorical columns (string/object)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical cols:", len(cat_cols))
print("Numeric cols:", len(num_cols))

# Indices for catboost Pool
cat_feature_indices = [X.columns.get_loc(c) for c in cat_cols]
cat_cols[:10], cat_feature_indices[:10]

Categorical cols: 32
Numeric cols: 14


(['ID',
  'country',
  'attitude_stable_business_environment',
  'attitude_worried_shutdown',
  'compliance_income_tax',
  'perception_insurance_doesnt_cover_losses',
  'perception_cannot_afford_insurance',
  'motor_vehicle_insurance',
  'has_mobile_money',
  'current_problem_cash_flow'],
 [0, 1, 3, 4, 5, 6, 7, 12, 13, 14])